# Notebook 02 — Feature Engineering

**Goal:** Compute all behavioural features from the enriched scrobble data and produce a user-level feature matrix for clustering.

**Inputs** (from `data/processed/`, all CSV):
- `scrobbles_updated.csv`
- `profiles.csv`
- `artist_genres.csv`
- `audio_features.csv`

**Outputs:**
- `data/processed/user_features.csv` — one row per user, all features

**Features computed:**

| Group | Features |
|---|---|
| Artist diversity | unique_artists, artist_entropy, artist_concentration_20 |
| Genre diversity | unique_genres, genre_entropy, genre_concentration_5, avg_genre_tags_per_play |
| Engagement | total_scrobbles, unique_tracks, track_replay_rate, avg_tracks_per_session, session_count |
| Discovery | discovery_velocity_30d/90d, novelty_ratio, top_artist_play_share |
| Temporal | temporal_hour/dow/month_entropy, morning/evening/weekend_ratio, temporal_stability_pc1..5 |
| Audio profile | mean danceability, energy, valence, tempo, acousticness, instrumentalness |

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd
from src.data.loader import load_artist_genres_csv

# With ~10,000 sampled users the CSV files are small enough to load directly.
scrobbles     = pd.read_csv('../data/processed/scrobbles_updated.csv',
                             parse_dates=['timestamp'])
profiles      = pd.read_csv('../data/processed/profiles.csv')
# load_artist_genres_csv deserialises the JSON-encoded 'genres' column to list[str]
artist_genres = load_artist_genres_csv('../data/processed/artist_genres.csv')
audio_features = pd.read_csv('../data/processed/audio_features.csv')

print(f'Scrobbles     : {len(scrobbles):,} rows, {scrobbles["userid"].nunique():,} users')
print(f'Artist genres : {len(artist_genres):,} artists, '
      f'{artist_genres["spotify_artist_id"].notna().sum():,} matched')
print(f'Audio features: {len(audio_features):,} tracks')

## 1. Artist Diversity Features

In [ ]:
from src.features.diversity import compute_artist_diversity

artist_div = compute_artist_diversity(scrobbles, top_n=20)
print(f'Artist diversity features: {artist_div.shape}')
artist_div.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
artist_div['unique_artists'].hist(ax=axes[0], bins=30)
axes[0].set_title('Unique Artists per User')

artist_div['artist_entropy'].hist(ax=axes[1], bins=30)
axes[1].set_title('Artist Shannon Entropy')

artist_div['artist_concentration_20'].hist(ax=axes[2], bins=30)
axes[2].set_title('Top-20 Artist Concentration')

plt.tight_layout()
plt.show()

## 2. Genre Diversity Features

In [ ]:
from src.features.diversity import compute_genre_diversity

genre_div = compute_genre_diversity(scrobbles, artist_genres, top_n=5)
print(f'Genre diversity features: {genre_div.shape}')
genre_div.describe()

## 3. Temporal Features

Includes PCA-compressed hourly listening profiles to capture **listening pattern stability** while avoiding multicollinearity across 24 hour-of-day columns.

In [ ]:
from src.features.temporal import compute_temporal_features

temporal = compute_temporal_features(scrobbles, pca_components=5)
print(f'Temporal features: {temporal.shape}')
temporal.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
temporal['temporal_hour_entropy'].hist(ax=axes[0], bins=20)
axes[0].set_title('Hour-of-Day Entropy')

temporal['weekend_ratio'].hist(ax=axes[1], bins=20)
axes[1].set_title('Weekend Listen Ratio')

temporal['avg_daily_plays'].hist(ax=axes[2], bins=30)
axes[2].set_title('Avg Daily Plays')

plt.tight_layout()
plt.show()

## 4. Engagement & Discovery Features

In [ ]:
from src.features.engagement import compute_engagement_features

engagement = compute_engagement_features(scrobbles, session_gap_minutes=30)
print(f'Engagement features: {engagement.shape}')
engagement.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(['track_replay_rate', 'avg_tracks_per_session',
                          'discovery_velocity_30d', 'novelty_ratio',
                          'top_artist_play_share', 'session_count']):
    engagement[col].hist(ax=axes[i], bins=25)
    axes[i].set_title(col.replace('_', ' ').title())

plt.tight_layout()
plt.show()

## 5. Audio Feature Profile

In [ ]:
from src.features.engagement import compute_audio_feature_profile

audio_profile = compute_audio_feature_profile(scrobbles, audio_features)
print(f'Audio feature profile: {audio_profile.shape}')
audio_profile.head()

## 6. Build Final Feature Matrix

In [ ]:
import os

# Join all feature blocks (already computed above — no need to reload scrobbles).
feature_matrix = (
    artist_div
    .join(genre_div,     how='outer')
    .join(temporal,      how='outer')
    .join(engagement,    how='outer')
    .join(audio_profile, how='outer')
)

# Add demographics (descriptive only, not used as clustering inputs)
profiles_indexed = profiles.set_index('userid')[['gender', 'age', 'country']]
feature_matrix = feature_matrix.join(profiles_indexed, how='left').sort_index()

os.makedirs('../data/processed', exist_ok=True)
feature_matrix.to_csv('../data/processed/user_features.csv')

print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')
feature_matrix.head()

## 6b. EDA — Feature Matrix Descriptive Statistics

Inspect the shape, completeness, and distributions of every feature group before feeding the matrix into the clustering pipeline.

In [ ]:
# ── Shape & completeness ────────────────────────────────────────────
print(f'Feature matrix: {feature_matrix.shape[0]:,} users × {feature_matrix.shape[1]} columns')
print(f'Index name: {feature_matrix.index.name}')
print()

missing = feature_matrix.isnull().sum()
missing_pct = (missing / len(feature_matrix) * 100).round(2)
completeness = (
    missing_pct
    .rename('missing_%')
    .to_frame()
    .query('`missing_%` > 0')
    .sort_values('missing_%', ascending=False)
)
if completeness.empty:
    print('No missing values — feature matrix is complete.')
else:
    print('Columns with missing values:')
    print(completeness.to_string())


In [ ]:
# ── Descriptive statistics by feature group ─────────────────────────

feature_groups = {
    'Diversity': [
        'unique_artists', 'artist_entropy', 'artist_concentration_20',
        'unique_genres', 'genre_entropy', 'genre_concentration_5',
        'avg_genre_tags_per_play',
    ],
    'Temporal': [
        'temporal_hour_entropy', 'temporal_dow_entropy', 'temporal_month_entropy',
        'morning_ratio', 'evening_ratio', 'weekend_ratio',
        'listening_days', 'avg_daily_plays',
    ],
    'Engagement': [
        'total_scrobbles', 'unique_tracks', 'track_replay_rate',
        'avg_tracks_per_session', 'session_count', 'avg_session_length_min',
        'discovery_velocity_30d', 'discovery_velocity_90d',
        'novelty_ratio', 'top_artist_play_share',
    ],
    'Audio (mean)': [
        'mean_danceability', 'mean_energy', 'mean_valence', 'mean_tempo',
        'mean_acousticness', 'mean_instrumentalness',
        'mean_liveness', 'mean_speechiness',
    ],
}

for group_name, cols in feature_groups.items():
    present = [c for c in cols if c in feature_matrix.columns]
    if not present:
        continue
    print(f'\n{'='*64}')
    print(f'  {group_name.upper()} FEATURES  ({len(present)} columns)')
    print(f'{'='*64}')
    stats = (
        feature_matrix[present]
        .describe(percentiles=[.05, .25, .50, .75, .95])
        .round(4)
    )
    print(stats.to_string())


In [ ]:
# ── Skewness & outlier summary ──────────────────────────────────────
import numpy as np

numeric = feature_matrix.select_dtypes(include=[np.number])

skew_df = numeric.skew().rename('skewness').to_frame()
skew_df['kurt'] = numeric.kurt()

# Flag columns where |skew| > 2 (may benefit from log transform)
skew_df['highly_skewed'] = skew_df['skewness'].abs() > 2

# IQR-based outlier rate per column
q1 = numeric.quantile(0.25)
q3 = numeric.quantile(0.75)
iqr = q3 - q1
outlier_rate = (
    ((numeric < (q1 - 1.5 * iqr)) | (numeric > (q3 + 1.5 * iqr)))
    .mean()
    .rename('outlier_rate_%')
    * 100
).round(2)

summary = skew_df.join(outlier_rate).sort_values('skewness', ascending=False)
print('Skewness, kurtosis, and IQR-outlier rate per feature:')
print(summary.round(3).to_string())
print(f'\nHighly skewed columns (|skew| > 2): {skew_df["highly_skewed"].sum()}')


In [ ]:
# ── Distribution plots — one panel per feature group ────────────────
import matplotlib.pyplot as plt

for group_name, cols in feature_groups.items():
    present = [c for c in cols if c in feature_matrix.columns]
    if not present:
        continue
    n = len(present)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows))
    axes = axes.flatten() if n > 1 else [axes]
    fig.suptitle(f'{group_name} Features — Distributions', fontsize=14, y=1.01)

    for ax, col in zip(axes, present):
        data = feature_matrix[col].dropna()
        ax.hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
        ax.axvline(data.median(), color='red', linewidth=1.2, linestyle='--',
                   label=f'median={data.median():.3f}')
        ax.set_title(col, fontsize=9)
        ax.legend(fontsize=7)
        ax.tick_params(labelsize=7)

    # Hide unused axes
    for ax in axes[len(present):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Pairwise correlations within each feature group ─────────────────
import matplotlib.pyplot as plt
import numpy as np

for group_name, cols in feature_groups.items():
    present = [c for c in cols if c in feature_matrix.columns]
    if len(present) < 2:
        continue
    corr = feature_matrix[present].corr()

    fig, ax = plt.subplots(figsize=(max(6, len(present)), max(5, len(present) - 1)))
    im = ax.imshow(corr.values, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_xticks(range(len(present)))
    ax.set_yticks(range(len(present)))
    ax.set_xticklabels(present, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(present, fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f'{group_name} — Correlation Matrix', fontsize=12)

    # Annotate cells
    for i in range(len(present)):
        for j in range(len(present)):
            ax.text(j, i, f'{corr.values[i, j]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(corr.values[i, j]) > 0.6 else 'black')
    plt.tight_layout()
    plt.show()

    # Flag high correlations
    high = (
        corr.abs()
        .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .loc[lambda s: s > 0.80]
        .sort_values(ascending=False)
    )
    if not high.empty:
        print(f'{group_name} — pairs with |r| > 0.80:')
        print(high.round(3).to_string())
        print()


## 6c. Key Variable Deep Dive

### Why these variables?

Twelve variables were selected as the analytical core of this study.
Each one captures a **distinct, interpretable dimension** of listening behaviour
and is expected to vary meaningfully across listener types.
Together they form a behavioural fingerprint — answering not just *what* a user
listens to, but *how*, *when*, and *how adventurously* they engage with music.

| Dimension | Variables | What they answer |
|---|---|---|
| **Volume & engagement** | `total_scrobbles`, `unique_tracks`, `track_replay_rate`, `avg_tracks_per_session` | How much does this person listen, and do they re-listen or always explore? |
| **Diversity & taste breadth** | `unique_artists`, `artist_entropy`, `genre_entropy`, `artist_concentration_20` | Is taste narrow or wide? Focused or scattered? |
| **Discovery & novelty** | `novelty_ratio`, `discovery_velocity_30d` | Is this person constantly finding new music, or sticking to established favourites? |
| **Temporal habits** | `temporal_hour_entropy`, `weekend_ratio` | Do they have a routine? Are they a daytime or night-time listener? |
| **Sonic signature** | `mean_energy`, `mean_valence` | What emotional and physical energy does their music have on average? |

> **Note on audio features:** `mean_energy` and `mean_valence` require a Spotify
> track match; users who primarily listen to obscure or non-English artists will
> have `NaN` for these columns — see the missing-data analysis below.

In [ ]:
# ── Engagement variables ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

KEY_ENGAGEMENT = {
    'total_scrobbles'       : 'Total Scrobbles',
    'unique_tracks'         : 'Unique Tracks Played',
    'track_replay_rate'     : 'Track Replay Rate\n(plays ÷ unique tracks)',
    'avg_tracks_per_session': 'Avg Tracks per Session',
}

present = {k: v for k, v in KEY_ENGAGEMENT.items() if k in feature_matrix.columns}
fig, axes = plt.subplots(2, len(present), figsize=(4.5 * len(present), 9))

for col_i, (col, label) in enumerate(present.items()):
    data = feature_matrix[col].dropna()
    skew = data.skew()

    # ── top row: histogram ───────────────────────────────────────────────────
    ax = axes[0, col_i]
    ax.hist(data, bins=60, color='#4C78A8', edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson', lw=1.5, ls='--', label=f'mean  {data.mean():.1f}')
    ax.axvline(data.median(), color='darkorange', lw=1.5, ls=':',  label=f'med   {data.median():.1f}')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_ylabel('Users')
    ax.legend(fontsize=7.5)
    ax.text(0.97, 0.94, f'skew = {skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8.5,
            color='darkred' if abs(skew) > 2 else '#333333',
            fontweight='bold' if abs(skew) > 2 else 'normal')

    # ── bottom row: boxplot (outliers as red dots) ───────────────────────────
    ax2 = axes[1, col_i]
    bp = ax2.boxplot(data, vert=True, patch_artist=True,
                     flierprops=dict(marker='.', color='crimson', markersize=3, alpha=0.5),
                     medianprops=dict(color='darkorange', linewidth=2),
                     boxprops=dict(facecolor='#4C78A8', alpha=0.5))
    ax2.set_ylabel('Value')
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    n_out = ((data < q1 - 1.5*(q3-q1)) | (data > q3 + 1.5*(q3-q1))).sum()
    ax2.set_title(f'{n_out} IQR outliers  ({n_out/len(data)*100:.1f}%)', fontsize=9)

plt.suptitle('Engagement Features — Distribution & Outliers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Printed observations ─────────────────────────────────────────────────────
print('ENGAGEMENT — KEY OBSERVATIONS\n' + '─'*60)
for col, label in present.items():
    data = feature_matrix[col].dropna()
    miss = feature_matrix[col].isnull().mean() * 100
    skew = data.skew()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr   = q3 - q1
    n_out = ((data < q1-1.5*iqr) | (data > q3+1.5*iqr)).sum()
    p95   = data.quantile(0.95)
    p50   = data.median()

    print(f'\n{label}')
    print(f'  n={len(data):,}  missing={miss:.1f}%  '
          f'median={p50:.1f}  mean={data.mean():.1f}  p95={p95:.1f}  max={data.max():.1f}')
    flags = []
    if abs(skew) > 2:
        flags.append(f'highly right-skewed (skew={skew:.2f}) — '
                     'power-law behaviour; consider log1p transform before clustering')
    if n_out / len(data) > 0.05:
        flags.append(f'{n_out/len(data)*100:.1f}% IQR outliers — '
                     'heavy tail; top users will pull cluster centroids')
    if p95 / (p50+1e-9) > 5:
        flags.append(f'p95/median ratio = {p95/(p50+1e-9):.1f}× — '
                     'extreme spread between typical and heavy users')
    if miss > 1:
        flags.append(f'{miss:.1f}% missing — check pipeline completeness')
    for f in flags:
        print(f'  ⚠  {f}')
    if not flags:
        print('  ✓  No major distribution issues')


In [ ]:
# ── Diversity & Discovery variables ─────────────────────────────────────────
KEY_DIVERSITY = {
    'unique_artists'         : 'Unique Artists',
    'artist_entropy'         : 'Artist Entropy\n(Shannon, bits)',
    'genre_entropy'          : 'Genre Entropy\n(Shannon, bits)',
    'artist_concentration_20': 'Artist Concentration\n(top-20 share)',
    'novelty_ratio'          : 'Novelty Ratio\n(new-artist plays, last 90d)',
    'discovery_velocity_30d' : 'Discovery Velocity\n(new artists/day, 30d)',
}

present = {k: v for k, v in KEY_DIVERSITY.items() if k in feature_matrix.columns}
ncols = 3
nrows = int(np.ceil(len(present) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
axes = axes.flatten()

for i, (col, label) in enumerate(present.items()):
    data = feature_matrix[col].dropna()
    skew = data.skew()
    ax = axes[i]

    # Colour-code: bounded [0,1] vars in teal, count/entropy vars in blue
    colour = '#2CA02C' if data.max() <= 1.01 else '#4C78A8'
    ax.hist(data, bins=50, color=colour, edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson',     lw=1.5, ls='--', label=f'mean  {data.mean():.3f}')
    ax.axvline(data.median(), color='darkorange',  lw=1.5, ls=':',  label=f'med   {data.median():.3f}')
    ax.set_title(label, fontsize=9.5, fontweight='bold')
    ax.set_ylabel('Users')
    ax.legend(fontsize=7)
    ax.text(0.97, 0.94, f'skew={skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            color='darkred' if abs(skew) > 2 else '#333333')

for ax in axes[len(present):]:
    ax.set_visible(False)

plt.suptitle('Diversity & Discovery Features — Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Printed observations ─────────────────────────────────────────────────────
print('DIVERSITY & DISCOVERY — KEY OBSERVATIONS\n' + '─'*60)

# Check for bimodality via dip in the histogram
def _is_bimodal_hint(data, bins=40):
    counts, edges = np.histogram(data, bins=bins)
    # Simple heuristic: find if there are two local maxima with a valley ≥30% between them
    from scipy.signal import argrelmax
    peaks = argrelmax(counts, order=3)[0]
    return len(peaks) >= 2

for col, label in present.items():
    data = feature_matrix[col].dropna()
    miss = feature_matrix[col].isnull().mean() * 100
    skew = data.skew()
    bimodal = _is_bimodal_hint(data)

    print(f'\n{label}')
    print(f'  n={len(data):,}  missing={miss:.1f}%  '
          f'min={data.min():.3f}  median={data.median():.3f}  '
          f'mean={data.mean():.3f}  max={data.max():.3f}')

    flags = []
    if bimodal:
        flags.append('possible bimodal distribution — two listener archetypes may exist '
                     '(e.g., dedicated explorers vs. loyal listeners)')
    if abs(skew) > 2:
        flags.append(f'highly skewed (skew={skew:.2f})')
    if miss > 5:
        flags.append(f'{miss:.1f}% missing — likely users with no Spotify-matched artists')
    if col == 'novelty_ratio' and (data == 0).mean() > 0.1:
        flags.append(f'{(data==0).mean()*100:.1f}% users have novelty_ratio=0 '
                     '(pure loyalists — listen only to already-known artists)')
    if col == 'novelty_ratio' and (data == 1).mean() > 0.05:
        flags.append(f'{(data==1).mean()*100:.1f}% users have novelty_ratio=1 '
                     '(could be sparse history artefact — users with <90d of data)')
    if col == 'discovery_velocity_30d' and data.quantile(0.95) > data.median() * 10:
        flags.append('extreme outliers — a small number of users discover many artists rapidly; '
                     'check whether these are real listeners or bots')
    for f in flags:
        print(f'  ⚠  {f}')
    if not flags:
        print('  ✓  No major issues')


In [ ]:
# ── Temporal variables ───────────────────────────────────────────────────────
KEY_TEMPORAL = {
    'temporal_hour_entropy': 'Hour-of-Day Entropy\n(bits — higher = listens at all hours)',
    'weekend_ratio'        : 'Weekend Ratio\n(fraction of plays Sat–Sun)',
    'morning_ratio'        : 'Morning Ratio\n(06:00–12:00)',
    'evening_ratio'        : 'Evening Ratio\n(18:00–00:00)',
    'avg_daily_plays'      : 'Avg Daily Plays',
}

present = {k: v for k, v in KEY_TEMPORAL.items() if k in feature_matrix.columns}
fig, axes = plt.subplots(1, len(present), figsize=(4.5 * len(present), 4.5))

# Reference lines for ratio variables
RATIO_BASELINE = {
    'weekend_ratio' : (2/7, '2/7 uniform baseline'),
    'morning_ratio' : (6/24, '6/24 uniform baseline'),
    'evening_ratio' : (6/24, '6/24 uniform baseline'),
}

for i, (col, label) in enumerate(present.items()):
    data = feature_matrix[col].dropna()
    ax = axes[i] if len(present) > 1 else axes

    ax.hist(data, bins=50, color='#9467BD', edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson',    lw=1.5, ls='--', label=f'mean  {data.mean():.3f}')
    ax.axvline(data.median(), color='darkorange', lw=1.5, ls=':',  label=f'med   {data.median():.3f}')

    if col in RATIO_BASELINE:
        ref, ref_label = RATIO_BASELINE[col]
        ax.axvline(ref, color='grey', lw=1.2, ls=(0,(5,5)), label=ref_label)

    ax.set_title(label, fontsize=9.5, fontweight='bold')
    ax.set_ylabel('Users')
    ax.legend(fontsize=7)
    skew = data.skew()
    ax.text(0.97, 0.94, f'skew={skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            color='darkred' if abs(skew) > 2 else '#333333')

plt.suptitle('Temporal Features — Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Printed observations ─────────────────────────────────────────────────────
print('TEMPORAL — KEY OBSERVATIONS\n' + '─'*60)

for col, label in present.items():
    data = feature_matrix[col].dropna()
    miss = feature_matrix[col].isnull().mean() * 100
    print(f'\n{label}')
    print(f'  n={len(data):,}  missing={miss:.1f}%  '
          f'median={data.median():.3f}  mean={data.mean():.3f}  std={data.std():.3f}')

    flags = []
    if col == 'weekend_ratio':
        baseline = 2/7
        above = (data > 0.50).mean() * 100
        below = (data < 0.15).mean() * 100
        if data.mean() > baseline * 1.1:
            flags.append(f'mean ({data.mean():.3f}) > uniform baseline ({baseline:.3f}) — '
                         'dataset skews toward weekend listeners')
        flags.append(f'{above:.1f}% of users have weekend_ratio > 0.50 (weekend-dominant)')
        flags.append(f'{below:.1f}% of users have weekend_ratio < 0.15 (weekday-dominant)')
    if col == 'temporal_hour_entropy':
        max_entropy = np.log2(24)
        pct_max = data.mean() / max_entropy * 100
        flags.append(f'users average {pct_max:.0f}% of maximum possible hour entropy — '
                     ('listening is fairly spread across the day'
                      if pct_max > 60 else 'most users have a clear daily listening schedule'))
    if col == 'avg_daily_plays' and data.skew() > 2:
        flags.append(f'right-skewed (skew={data.skew():.2f}) — '
                     'heavy users inflate the mean; prefer median for reporting')
    if miss > 1:
        flags.append(f'{miss:.1f}% missing')
    for f in flags:
        print(f'  ⚠  {f}' if 'dominant' not in f and 'spread' not in f else f'  →  {f}')
    if not flags:
        print('  ✓  No major issues')


In [ ]:
# ── Audio feature variables ──────────────────────────────────────────────────
KEY_AUDIO = {
    'mean_energy'          : 'Mean Energy\n(0=low, 1=high)',
    'mean_valence'         : 'Mean Valence\n(0=sad, 1=happy)',
    'mean_danceability'    : 'Mean Danceability',
    'mean_acousticness'    : 'Mean Acousticness',
    'mean_instrumentalness': 'Mean Instrumentalness',
}

present = {k: v for k, v in KEY_AUDIO.items() if k in feature_matrix.columns}
fig, axes = plt.subplots(1, len(present), figsize=(4.2 * len(present), 4.5))

for i, (col, label) in enumerate(present.items()):
    data = feature_matrix[col].dropna()
    ax = axes[i] if len(present) > 1 else axes
    skew = data.skew()

    ax.hist(data, bins=50, color='#E45756', edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(data.mean(),   color='navy',       lw=1.5, ls='--', label=f'mean  {data.mean():.3f}')
    ax.axvline(data.median(), color='darkorange', lw=1.5, ls=':',  label=f'med   {data.median():.3f}')
    # Spotify's all-tracks averages for reference
    SPOTIFY_GLOBAL = {'mean_energy': 0.64, 'mean_valence': 0.51,
                      'mean_danceability': 0.67, 'mean_acousticness': 0.25}
    if col in SPOTIFY_GLOBAL:
        ax.axvline(SPOTIFY_GLOBAL[col], color='grey', lw=1.1, ls=(0,(4,4)),
                   label=f'Spotify avg {SPOTIFY_GLOBAL[col]:.2f}')
    ax.set_title(label, fontsize=9.5, fontweight='bold')
    ax.set_ylabel('Users')
    ax.legend(fontsize=7)
    ax.text(0.97, 0.94, f'skew={skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8)

plt.suptitle('Audio Feature Profile — User Means', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Missing-data analysis for audio features ─────────────────────────────────
print('AUDIO FEATURES — KEY OBSERVATIONS\n' + '─'*60)

all_audio_cols = [c for c in [
    'mean_energy','mean_valence','mean_danceability',
    'mean_acousticness','mean_instrumentalness','mean_liveness','mean_speechiness'
] if c in feature_matrix.columns]

if all_audio_cols:
    # Users with ALL audio features missing
    all_missing_mask = feature_matrix[all_audio_cols].isnull().all(axis=1)
    print(f'\nUsers with NO audio features at all: '
          f'{all_missing_mask.sum():,} ({all_missing_mask.mean()*100:.1f}%)')
    print('  ⚠  These users listen predominantly to artists not found on Spotify.')
    print('       They will be imputed with median values before clustering — '
          'their audio-feature dimensions carry no real signal.')

    # Per-column missing rate
    print(f'\nPer-column missing rate:')
    for col in all_audio_cols:
        miss = feature_matrix[col].isnull().mean() * 100
        print(f'  {col:<28} {miss:5.1f}% missing')

    # Distribution shape flags
    print()
    for col, label in present.items():
        data = feature_matrix[col].dropna()
        skew = data.skew()
        flags = []
        if data.mean() < 0.35:
            flags.append(f'users lean low ({data.mean():.2f}) — '
                         'this cohort may skew toward acoustic / low-energy music')
        if data.mean() > 0.70:
            flags.append(f'users lean high ({data.mean():.2f}) — '
                         'cohort skews toward high-energy / danceable music')
        if col == 'mean_instrumentalness' and data.median() < 0.05:
            flags.append('median near zero — most users listen to vocal tracks; '
                         'classical/ambient listeners will be a small tail')
        if abs(skew) > 1.5:
            flags.append(f'skew={skew:.2f} — not normally distributed; '
                         'distribution shape may itself be a clustering signal')
        if flags:
            print(f'{label.split(chr(10))[0]}:')
            for f in flags:
                print(f'  →  {f}')


In [ ]:
# ── Consolidated issues & recommendations ────────────────────────────────────
import pandas as pd
import numpy as np

ALL_KEY_VARS = [
    'total_scrobbles', 'unique_tracks', 'track_replay_rate', 'avg_tracks_per_session',
    'unique_artists', 'artist_entropy', 'genre_entropy', 'artist_concentration_20',
    'novelty_ratio', 'discovery_velocity_30d',
    'temporal_hour_entropy', 'weekend_ratio',
    'mean_energy', 'mean_valence',
]
present = [c for c in ALL_KEY_VARS if c in feature_matrix.columns]

rows = []
for col in present:
    data = feature_matrix[col].dropna()
    miss = feature_matrix[col].isnull().mean() * 100
    skew = data.skew()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    n_out = int(((data < q1-1.5*iqr) | (data > q3+1.5*iqr)).sum())
    out_pct = n_out / len(feature_matrix) * 100

    issues = []
    if miss > 5:    issues.append('HIGH MISSING')
    elif miss > 1:  issues.append('some missing')
    if abs(skew) > 3: issues.append('extreme skew')
    elif abs(skew) > 2: issues.append('high skew')
    if out_pct > 10: issues.append('many outliers')
    elif out_pct > 5: issues.append('some outliers')

    recommendation = ''
    if 'extreme skew' in issues or 'high skew' in issues:
        recommendation = 'log1p transform'
    if 'HIGH MISSING' in issues:
        recommendation = ('median impute — but note bias' if recommendation == ''
                          else recommendation + ' + median impute')
    if recommendation == '' and not issues:
        recommendation = 'ready as-is'

    rows.append({
        'Variable'      : col,
        'n_valid'       : len(data),
        'missing_%'     : round(miss, 1),
        'median'        : round(data.median(), 3),
        'mean'          : round(data.mean(), 3),
        'skewness'      : round(skew, 2),
        'outliers_%'    : round(out_pct, 1),
        'issues'        : ', '.join(issues) if issues else '—',
        'recommendation': recommendation,
    })

summary_df = pd.DataFrame(rows).set_index('Variable')

# Highlight rows with issues
def _highlight(row):
    colour = ''
    if 'HIGH MISSING' in row['issues'] or 'extreme skew' in row['issues']:
        colour = 'background-color: #ffe0e0'
    elif row['issues'] != '—':
        colour = 'background-color: #fff7d6'
    return [colour] * len(row)

print('VARIABLE HEALTH SUMMARY')
print('Rows highlighted: red = major issue, yellow = minor issue\n')
try:
    display(summary_df.style.apply(_highlight, axis=1))
except NameError:
    print(summary_df.to_string())

print('\nKEY ACTIONS before clustering:')
log_candidates = [c for c in present
                  if feature_matrix[c].dropna().skew() > 2]
if log_candidates:
    print(f'  1. Apply log1p transform to: {log_candidates}')
print('  2. Impute remaining NaNs with column median (SimpleImputer)')
print('  3. StandardScaler — applied in clustering pipeline')
print('  4. PCA (95% variance) — will further compress correlated and skewed dims')
print('  5. Revisit users with 0 audio features post-clustering: '
      'they may form a spurious cluster driven purely by imputed medians')


## 7. Correlation Analysis & Multicollinearity Check

In [ ]:
# Select numeric clustering features (exclude demographics)
clustering_features = feature_matrix.select_dtypes(include=[np.number]).drop(
    columns=['age'], errors='ignore'
)

corr = clustering_features.corr()

plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='RdBu', center=0, vmin=-1, vmax=1,
    annot=False, fmt='.2f', linewidths=0.3,
)
plt.title('Feature Correlation Matrix (Clustering Features)')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Pairs with |r| > 0.75
high_corr = (
    corr.abs().where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack().reset_index()
    .rename(columns={0: 'correlation', 'level_0': 'feat_a', 'level_1': 'feat_b'})
    .query('correlation > 0.75')
    .sort_values('correlation', ascending=False)
)
print(f'Highly correlated pairs (|r|>0.75): {len(high_corr)}')
print(high_corr.to_string())

> Note: Remaining multicollinearity is handled at the clustering stage via PCA dimensionality reduction before fitting KMeans/HDBSCAN.

Proceed to **Notebook 03** for clustering.